In [1]:
# !pip install pytorch-lightning

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
# from torch_geometric.nn import GCNConv
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset, Dataset
import random, math
from torch.utils.data import random_split
from pytorch_lightning import Trainer

from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [3]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [4]:
NAME = 'movie'

N_STATIC_RELATION = 20
N_CLUSTER = 12

USER_START_ID = 1
USER_END_ID = 5433
ITEM_START_ID = 5434  # book: 2891
ITEM_END_ID = 10329
STATIC_ENTITY_START_ID = 10330  # book: 2891
STATIC_ENTITY_END_ID = 58529

N_ENTITY = STATIC_ENTITY_END_ID
N_RELATION = N_STATIC_RELATION + N_CLUSTER

In [5]:
class KnownledgeGraph():
    def __init__(self, graph_file, graph_neighbor_sample_size=1):
        self.graph_file = graph_file
        self.graph_neighbor_sample_size = graph_neighbor_sample_size
        self.graph_entities = None
        self.graph_relations = None
        self.graph_n_entity = 0
        self.graph_n_relation = 0
        self.graph = defaultdict(set)

    def build(self):
        df = pd.read_csv(self.graph_file)

        for head_id, relation_id, tail_id, _, _, _ in df.values:
            self.graph[head_id].add((tail_id, relation_id))
            self.graph[tail_id].add((head_id, relation_id))

    def get_neighbors(self, entity_id, sample_size=8):
        """Get neighbors for an entity with optional sampling"""
        if sample_size is None:
            sample_size = self.graph_neighbor_sample_size

        entity_id = entity_id.item() if torch.is_tensor(entity_id) else entity_id
        neighbors = list(self.graph.get(entity_id, set()))
        n_neighbors = len(neighbors)

        if n_neighbors == 0:
            return torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)

        if n_neighbors >= sample_size:
            sampled_indices = np.random.choice(n_neighbors, size=sample_size, replace=False)
        else:
            sampled_indices = np.random.choice(n_neighbors, size=sample_size, replace=True)

        sampled_neighbors = [neighbors[i] for i in sampled_indices]
        adj_entities = torch.tensor([n for n, _ in sampled_neighbors], dtype=torch.long)
        adj_relations = torch.tensor([r for _, r in sampled_neighbors], dtype=torch.long)

        return adj_entities, adj_relations

### NEW DATA MODULE

In [6]:
class RecommenderDataModule(pl.LightningDataModule):
    def __init__(self, train_df, val_df, test_df, graph: KnownledgeGraph, batch_size):
        super().__init__()
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self.batch_size = batch_size
        self.graph = graph

    def prepare_data(self):
        self.train_dataset = []
        for _, row in self.train_df.iterrows():
            user_id = row['user_id']
            user_item_relation = row['relation_id']
            item_id = row['entity_id']
            neighbor, nb_relation_id = self.graph.get_neighbors(item_id)
            self.train_dataset.append((user_id, user_item_relation, item_id, neighbor, nb_relation_id))

        self.val_dataset = []
        for _, row in self.val_df.iterrows():
            user_id = row['user_id']
            user_item_relation = row['relation_id']
            item_id = row['entity_id']
            neighbor, nb_relation_id = self.graph.get_neighbors(item_id)
            self.val_dataset.append((user_id, user_item_relation, item_id, neighbor, nb_relation_id))

        self.test_dataset = []
        for _, row in self.test_df.iterrows():
            user_id = row['user_id']
            user_item_relation = row['relation_id']
            item_id = row['entity_id']
            neighbor, nb_relation_id = self.graph.get_neighbors(item_id)
            self.test_dataset.append((user_id, user_item_relation, item_id, neighbor, nb_relation_id))


        train_user_pos_items = defaultdict(set)
        for u, r, i, _, _ in self.train_dataset:
            train_user_pos_items[u].add(i)

        val_user_pos_items = defaultdict(set)
        for u, r, i, _, _ in self.val_dataset:
            val_user_pos_items[u].add(i)

        test_user_pos_items = defaultdict(set)
        for u, r, i, _, _ in self.test_dataset:
            test_user_pos_items[u].add(i)

        self.train_user_pos_items = train_user_pos_items
        self.val_user_pos_items = val_user_pos_items
        self.test_user_pos_items = test_user_pos_items

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)
    
    # def test_dataloader(self):
    #     return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)

In [7]:
class SumAggregator(nn.Module):
    def __init__(self, embedding_dim):
        super(SumAggregator, self).__init__()
        self.linear = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, neighbor_embs, central_embs):
        """
        neighbor_embs: Tensor of shape (batch, emb_dim)
        central_embs: Tensor of shape (batch, emb_dim)
        """

        # Combine with central entity embedding
        combined = neighbor_embs + central_embs  # shape: (batch, emb_dim) ([1204, 64])

        # Linear + activation
        output = torch.tanh(self.linear(combined))  # shape: (batch, emb_dim) torch.Size([1204, 64])
        return output

In [ ]:
class KGCN(pl.LightningModule):
    def __init__(self, graph: KnownledgeGraph,
                 embedding_dim=64, lr=0.001, lambda_reg=1e-5):
        super().__init__()
        self.save_hyperparameters()

        model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_device = model_device

        # self.graph = graph
        # self.edge_index = None

        # self.num_users = num_users
        # self.num_items = num_items

        self.embedding_dim = embedding_dim

        # Embedding layers
        # self.user_embedding = nn.Embedding(num_users, embedding_dim).to(model_device)
        # self.entity_embedding = nn.Embedding(graph.graph_n_entity, embedding_dim).to(model_device)
        # self.relation_embedding = nn.Embedding(graph.graph_n_relation, embedding_dim).to(model_device)
        self.entity_embedding = nn.Embedding(N_ENTITY + 1, embedding_dim, padding_idx = 0).to(model_device)
        self.relation_embedding = nn.Embedding(N_RELATION + 1, embedding_dim, padding_idx = 0).to(model_device)

        # Aggregator
        self.aggregator = SumAggregator(embedding_dim).to(model_device)

        self.dropout = nn.Dropout(p=0.1)

    def setup(self, stage=None):
        self.train_user_pos_items = self.trainer.datamodule.train_user_pos_items
        self.val_user_pos_items = self.trainer.datamodule.val_user_pos_items
        self.test_user_pos_items = self.trainer.datamodule.test_user_pos_items

    @staticmethod
    def hit_at_k(pred_items, true_items, k):
        hits = 0
        for pred, true in zip(pred_items, true_items):
            # Count if any true item is in top-k predictions
            if len(set(pred[:k]) & set(true)) > 0:
                hits += 1
        return hits / len(true_items)

    @staticmethod
    def ndcg_at_k(pred_items, true_items, k):
        ndcg = 0.0
        for pred, true in zip(pred_items, true_items):
            gains = []
            for idx, item in enumerate(pred[:k]):
                gains.append(1 if item in true else 0)
            ideal_gains = [1] * min(len(true), k)
            dcg = sum(g / math.log2(i+2) for i, g in enumerate(gains))
            idcg = sum(g / math.log2(i+2) for i, g in enumerate(ideal_gains))
            ndcg += dcg / idcg if idcg > 0 else 0
        return ndcg / len(true_items)

    @staticmethod
    def recall_at_k(pred_items, true_items, k):
        recall = 0.0
        for pred, true in zip(pred_items, true_items):
            recall += len(set(pred[:k]) & set(true)) / len(true)
        return recall / len(true_items)

    @staticmethod
    def precision_at_k(pred_items, true_items, k):
        precision = 0.0
        for pred, true in zip(pred_items, true_items):
            precision += len(set(pred[:k]) & set(true)) / k
        return precision / len(true_items)

    def forward(self, batch):
        user_ids, relation_ids, item_ids, neighbor_ids, nb_relation_ids = batch


        # full_user_embs = self.entity_embedding.weight[USER_START_ID: USER_END_ID]
        full_entity_embs = self.entity_embedding.weight # torch.Size([58530, 128])
        # full_relation_embs = self.relation_embedding.weight

        user_embs = self.entity_embedding(user_ids) #torch.Size([512, 128])
        neighbor_embs = self.entity_embedding(neighbor_ids) # torch.Size([512, 8, 128])

        relation_embs = self.relation_embedding(relation_ids) # torch.Size([512, 8, 128])
        nb_relation_embs = self.relation_embedding(nb_relation_ids) # torch.Size([512, 8, 128])

        item_embs = self.entity_embedding(item_ids) #torch.Size([512, 128])

        ################ User - Relation Attention ################
        # Expand user_embs to match neighbor dimension
        user_relation_embs = user_embs + relation_embs
        user_relation_embs_expanded = user_relation_embs.unsqueeze(1)  # [1204, 1, 64]

        # Compute scores: dot product between relation_embs and user_embs
        # scores = torch.exp(-torch.abs(user_relation_embs_expanded - nb_relation_embs).sum(dim=2))  # [batch_size, k]
        user_relation_embs_norm = F.normalize(user_relation_embs, p=2, dim=1) # (batch, embed_dim) 
        nb_relation_embs_norm = F.normalize(nb_relation_embs, p=2, dim=2) # (batch, 8. embed_dim)
        # scores = (user_relation_embs_norm.unsqueeze(1) * nb_relation_embs_norm).sum(dim=2) # (batch, N)
        scores = torch.bmm(nb_relation_embs_norm, user_relation_embs_norm.unsqueeze(2)).squeeze(2) # (batch, N)

        attention_weights = F.softmax(scores, dim=1).unsqueeze(-1)  # [1204, 8, 1]
        weighted_neighbor_embs = neighbor_embs * attention_weights  # [1204, 8, 64]
        weighted_neighbor_embs = weighted_neighbor_embs.mean(dim=1) # [1204, 64]
        ################ User - Relation Attention ################

        updated_item_embs  = self.aggregator(weighted_neighbor_embs, item_embs) #([1204, 64])

        agg_full_entity_embs = full_entity_embs.clone()
        agg_full_entity_embs.index_copy_(0, item_ids, updated_item_embs)
        # agg_full_entity_embs.index_copy_(0, user_ids, user_embs) ### Don't need this because user_embs doesnt' change

        agg_full_entity_embs = self.dropout(agg_full_entity_embs)


        return agg_full_entity_embs

    def compute_loss(self, batch, agg_full_entity_embs):
        user_ids, relation_ids, item_ids, neighbor_ids, nb_relation_ids = batch

        user_embs = agg_full_entity_embs[user_ids]
        pos_item_embs = agg_full_entity_embs[item_ids]

        relation_embs = self.relation_embedding(relation_ids)
        user_relation_embs = user_embs + relation_embs

        
        # pos_scores = torch.exp(-torch.abs(user_relation_embs - pos_item_embs).sum(dim=1))
        user_relation_embs_norm = F.normalize(user_relation_embs, p=2, dim=1) #torch.Size([512, 64])
        pos_item_embs_norm = F.normalize(pos_item_embs, p=2, dim=1)
        pos_scores = (user_relation_embs_norm * pos_item_embs_norm).sum(dim=1)


        ####################### Hard negative Sampling #######################
        full_item_embs = agg_full_entity_embs[ITEM_START_ID: ITEM_END_ID + 1]    # torch.Size([4896, 128]) #[:self.num_items]

        # distances = torch.cdist(user_relation_embs, full_item_embs, p=1) # [batch_size, num_items] torch.Size([1204, 4779])
        # scores = torch.exp(-distances)  # [batch_size, num_items] torch.Size([1204, 4779])
        full_item_embs_norm = F.normalize(full_item_embs, p=2, dim=1) #torch.Size([4896, 64])
        scores = torch.matmul(user_relation_embs_norm, full_item_embs_norm.T)

        ######## Mask only those pos_id in batch, not all pos_item_ids of each user ########
        # for i, pos_id in enumerate(item_ids): #Score of positive item will be set to -inf so that they won't be picked by TopK
        #     scores[i, pos_id] = float('-inf')   # it is the score between the ith user in batch_size and positive item pos_id
        ######## Mask only those pos_id in batch, not all pos_item_ids of each user ########


        ######## Mask all pos_item_ids of the user in training_set ########
        for i, u in enumerate(user_ids.tolist()):
            pos_item_ids = [item - ITEM_START_ID for item in self.train_user_pos_items[u]]      # pos_item_ids was shift left by ITEM_START_ID to become zero base
            scores[i, pos_item_ids] = float('-inf')
        ######## Mask all pos_item_ids of the user in training_set ########

        k = 10 # Select top-K most negatives for each user
        neg_item_ids = torch.topk(scores, k=k, dim=1).indices  # [batch_size, k]
        neg_item_ids = neg_item_ids + ITEM_START_ID         # neg_item_ids is zero based, need to shift right by ITEM_START_ID

        # Get embeddings for these negatives
        neg_items_emb = agg_full_entity_embs[neg_item_ids]  # shape: [batch_size, k, emb_dim]

        # neg_scores = torch.exp(-torch.abs(user_relation_embs.unsqueeze(1) - neg_items_emb).sum(dim=2))  # [batch_size, k]
        neg_items_emb_norm = F.normalize(neg_items_emb, p=2, dim=2) # (batch, embed_dim)
        # neg_scores = (user_relation_embs_norm.unsqueeze(1) * neg_items_emb_norm).sum(dim=2) # (batch, N)
        neg_scores = torch.bmm(neg_items_emb_norm, user_relation_embs_norm.unsqueeze(2)).squeeze(2) # (batch, N)
        
        
        ####################### Hard negative Sampling #######################

        ####################### Compute Loss #######################
        # neg_scores = neg_scores.mean(dim=1) # [batch_size]
        # # scores = torch.cat([pos_scores, neg_scores], dim=0)
        # # labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)], dim=0)

        # # loss = F.binary_cross_entropy(scores, labels) # can't use BCE because pos_score and neg_score exceed 0, 1
        # loss =- F.logsigmoid(pos_scores - neg_scores).mean() #### chay duoc


        ##########InfoNCE Loss
        pos_scores = pos_scores.unsqueeze(1)  # Ép pos_scores từ [batch_size] thành [batch_size, 1]
        logits = torch.cat([pos_scores, neg_scores], dim=1)   # Ghép với neg_scores [batch_size, N]

        temperature = 0.05
        logits = logits / temperature         # Thêm Temperature để chống pos score = neg score (QUAN TRỌNG)

        labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
        loss = F.cross_entropy(logits, labels)
         ##########InfoNCE Loss
        ####################### Compute Loss #######################

        return loss, pos_scores, neg_scores.mean(dim=1)

    def training_step(self, batch, batch_idx):
        agg_full_entity_embs = self(batch)
        loss, _, _ = self.compute_loss(batch, agg_full_entity_embs) #torch.Size([58530, 128])

        self.log("train_loss", loss, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        user_ids, relation_ids, item_ids, neighbor_ids, nb_relation_ids = batch

        agg_full_entity_embs = self(batch)
        full_item_embs = agg_full_entity_embs[ITEM_START_ID: ITEM_END_ID + 1]      # [:self.num_items]

        user_embs = agg_full_entity_embs[user_ids]

        relation_embs = self.relation_embedding(relation_ids)
        user_relation_embs = user_embs + relation_embs

        # distances = torch.cdist(user_relation_embs, full_item_embs, p=1)  # [batch_size, num_items] torch.Size([1204, 4779]) 
        # scores = torch.exp(-distances)  # [batch_size, num_items] torch.Size([1204, 4779]), it is the score between the ith user in batch_size and ALL items
        user_relation_embs_norm = F.normalize(user_relation_embs, p=2, dim=1)
        full_item_embs_norm = F.normalize(full_item_embs, p=2, dim=1)
        scores = torch.matmul(user_relation_embs_norm, full_item_embs_norm.T)

        # ########## Mask those user-item pair that already in training set so that it won't suggest again
        mask = torch.zeros_like(scores, dtype=torch.bool)
        for i, u in enumerate(user_ids.tolist()):
            trained_items = [item - ITEM_START_ID  for item in self.train_user_pos_items[u]]  # was shift left by ITEM_START_ID to become zero base
            mask[i, trained_items] = True

        scores = scores.masked_fill(mask, float('-inf'))    #### Make them to -inf so that TopK won't pick again
        ########## Mask those user-item pair that already in training set so that it won't suggest again


        ################ Calculate metrics
        k_values = [10]  # Example: you can add more values as needed

        for k in k_values:
            # Get top-k items for this k
            topk_indices = torch.topk(scores, k=k, dim=1).indices # (1024, K=5)
            topk_items = (topk_indices + ITEM_START_ID).tolist()   # shift right by ITEM_START_ID to return origin id

            true_items = []  # (1024, variable length), each user may have multiple positive items
            for u in user_ids.tolist():
                adjusted_val_items = [item for item in self.val_user_pos_items[u]]
                true_items.append(adjusted_val_items)

            # Compute metrics for this k
            hit = self.hit_at_k(topk_items, true_items, k)
            ndcg = self.ndcg_at_k(topk_items, true_items, k)
            recall = self.recall_at_k(topk_items, true_items, k)
            precision = self.precision_at_k(topk_items, true_items, k)

            # Log metrics dynamically
            self.log(f"val_hit@{k:02d}", hit, prog_bar=True)
            self.log(f"val_recall@{k:02d}", recall, prog_bar=True)
            self.log(f"val_precision@{k:02d}", precision, prog_bar=True)
            self.log(f"val_ndcg@{k:02d}", ndcg, prog_bar=True)


        # Compute bpr_loss
        loss,  pos_scores, neg_scores = self.compute_loss(batch, agg_full_entity_embs)

        self.log('val_loss', loss, prog_bar=True, logger=True)

        # Print first 10 logits
        # if batch_idx == 0:
        #     print("pos: ", pos_scores[:10])
        #     print("neg: ", neg_scores[:10])

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.75)
        return [optimizer], [scheduler]

## Main

In [9]:
graph = KnownledgeGraph(graph_file = f'./data/{NAME}/{NAME}_processed_static_graph.csv')
graph.build()


train_df = pd.read_csv(f'./data/{NAME}/{NAME}_gmm_train_interactions.csv')
val_df = pd.read_csv(f'./data/{NAME}/{NAME}_gmm_val_interactions.csv')
test_df = pd.read_csv(f'./data/{NAME}/{NAME}_gmm_test_interactions.csv')
data_module = RecommenderDataModule(train_df, val_df, test_df, graph, batch_size= 512)
data_module.prepare_data()

In [10]:
models = {
    'KGCN': KGCN(
        graph = graph,
        embedding_dim= 64,
        lr= 0.025
    )
}

for model_name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")

    # Early stopping callback
    early_stop_callback = EarlyStopping(
        monitor="val_loss",     # metric to monitor
        patience=50,            # number of epochs with no improvement after which training will be stopped
        mode="min",             # 'min' because we want to minimize val_loss
        verbose=True
    )

    checkpoint_loss = ModelCheckpoint(
            monitor="val_loss",
            dirpath="./checkpoints",
            filename=f"{model_name}-{NAME}-{{epoch:02d}}-{{val_loss:.3f}}",
            save_top_k=1,
            mode="min",
    )

    checkpoint_hit = ModelCheckpoint(
            monitor="val_hit@10",
            dirpath="./checkpoints",
            filename=f"{model_name}-{NAME}-{{epoch:02d}}-{{val_hit@10:.3f}}",
            save_top_k=1,
            mode="max",
    )

    checkpoint_recall = ModelCheckpoint(
            monitor="val_recall@10",
            dirpath="./checkpoints",
            filename=f"{model_name}-{NAME}-{{epoch:02d}}-{{val_recall@10:.3f}}",
            save_top_k=1,
            mode="max",
    )

    checkpoint_ndcg = ModelCheckpoint(
            monitor="val_ndcg@10",
            dirpath="./checkpoints",
            filename=f"{model_name}-{NAME}-{{epoch:02d}}-{{val_loss:.3f}}{{val_ndcg@10:.3f}}",
            save_top_k=1,
            mode="max",
    )

    checkpoint_precision = ModelCheckpoint(
            monitor="val_precision@10",
            dirpath="./checkpoints",
            filename=f"{model_name}-{NAME}-{{epoch:02d}}-{{val_precision@10:.3f}}",
            save_top_k=1,
            mode="max",
    )

    trainer = Trainer(
            num_sanity_val_steps=0,
            max_epochs=500,
            accelerator="auto",
            callbacks=[checkpoint_loss, checkpoint_hit, checkpoint_recall, checkpoint_precision, checkpoint_ndcg, early_stop_callback],
            # callbacks=[GradientCheckCallback(), early_stop_callback],
        )

    trainer.fit(model, data_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores



Training KGCN



  | Name               | Type          | Params | Mode  | FLOPs
---------------------------------------------------------------------
0 | entity_embedding   | Embedding     | 3.7 M  | train | 0    
1 | relation_embedding | Embedding     | 2.1 K  | train | 0    
2 | aggregator         | SumAggregator | 4.2 K  | train | 0    
3 | dropout            | Dropout       | 0      | train | 0    
---------------------------------------------------------------------
3.8 M     Trainable params
0         Non-trainable params
3.8 M     Total params
15.009    Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 0: 100%|██████████| 78/78 [00:11<00:00,  6.92it/s, v_num=5, train_loss_step=2.900, val_hit@10=0.106, val_recall@10=0.0314, val_precision@10=0.0127, val_ndcg@10=0.0221, val_loss=2.730, train_loss_epoch=5.550]

Metric val_loss improved. New best score: 2.735


...

Epoch 16: 100%|██████████| 78/78 [00:24<00:00,  3.18it/s, v_num=5, train_loss_step=2.630, val_hit@10=0.207, val_recall@10=0.0902, val_precision@10=0.0269, val_ndcg@10=0.0605, val_loss=2.460, train_loss_epoch=2.980]

Metric val_loss improved by 0.010 >= min_delta = 0.0. New best score: 2.456


Epoch 24:  50%|█████     | 39/78 [00:09<00:09,  4.20it/s, v_num=5, train_loss_step=3.090, val_hit@10=0.191, val_recall@10=0.0775, val_precision@10=0.0245, val_ndcg@10=0.0507, val_loss=2.470, train_loss_epoch=3.030]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [ ]:
# # ########### Validation
# checkpoint_path = "/content/KGCN-book-NB-51.ckpt"
# model = KGCN.load_from_checkpoint(checkpoint_path)

# trainer = Trainer(accelerator="gpu", devices=1)
# trainer.validate(model, datamodule=data_module)

# # for name, module in model.named_modules():
# #     print(f"{name}, {module}")
